# ClimaCidadão — previsão de risco térmico na próxima hora
**Pergunta de ML:** a partir das condições meteorológicas atuais da estação A701, é possível prever se a próxima hora terá risco térmico?

Modelo: Random Forest de classificação. Dados reais e públicos do INMET; nenhuma linha simulada.

In [ ]:
from pathlib import Path
import hashlib, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)


In [ ]:
files = sorted(Path('dados').glob('*.CSV')) + sorted(Path('dados').glob('*.csv'))
assert files, 'Baixe e extraia os CSVs oficiais conforme dados/README.md'
print('Arquivos:', len(files))
for p in files: print(p.name, hashlib.sha256(p.read_bytes()).hexdigest()[:16])

def read_inmet(path):
    raw = path.read_text(encoding='latin1', errors='replace').splitlines()
    header = next(i for i, line in enumerate(raw) if 'Data' in line and ';' in line)
    df = pd.read_csv(path, sep=';', decimal=',', encoding='latin1', skiprows=header)
    df.columns = [re.sub(r'[^a-z0-9]+','_', c.lower().encode('ascii','ignore').decode()).strip('_') for c in df.columns]
    return df
raw = pd.concat([read_inmet(p) for p in files], ignore_index=True)
print(raw.shape); display(raw.head())


In [ ]:
def find_col(words):
    for c in raw.columns:
        if all(w in c for w in words): return c
    raise KeyError(words)
date_col=find_col(['data']); hour_col=find_col(['hora']); temp_col=find_col(['temperatura','bulbo','horaria']); hum_col=find_col(['umidade','relativa','horaria']); press_col=find_col(['pressao','atmosferica','horaria']); wind_col=find_col(['vento','velocidade'])
df=pd.DataFrame({'date':raw[date_col], 'hour':raw[hour_col], 'temp':raw[temp_col], 'humidity':raw[hum_col], 'pressure':raw[press_col], 'wind':raw[wind_col]})
for c in ['temp','humidity','pressure','wind']: df[c]=pd.to_numeric(df[c],errors='coerce').replace(9999,np.nan)
df['datetime']=pd.to_datetime(df['date'].astype(str)+' '+df['hour'].astype(str).str.extract(r'(\d{2})',expand=False)+':00',errors='coerce',utc=True)
df=df.dropna().sort_values('datetime').drop_duplicates('datetime').reset_index(drop=True)
# Alvo pedagógico: próxima hora com temperatura >=32 C e umidade >=60%, proxy operacional definido pela equipe.
df['target']=((df['temp'].shift(-1)>=32)&(df['humidity'].shift(-1)>=60)).astype(int)
df=df.iloc[:-1].copy()
df['hour_sin']=np.sin(2*np.pi*df.datetime.dt.hour/24); df['hour_cos']=np.cos(2*np.pi*df.datetime.dt.hour/24)
df['month_sin']=np.sin(2*np.pi*df.datetime.dt.month/12); df['month_cos']=np.cos(2*np.pi*df.datetime.dt.month/12)
print(df.datetime.min(), df.datetime.max(), df.shape, df.target.value_counts(normalize=True)); display(df.describe())


In [ ]:
features=['temp','humidity','pressure','wind','hour_sin','hour_cos','month_sin','month_cos']
cut=int(len(df)*.8)
train,test=df.iloc[:cut],df.iloc[cut:]
X_train,y_train=train[features],train.target; X_test,y_test=test[features],test.target
assert train.datetime.max()<test.datetime.min(), 'Vazamento temporal'
print('Treino',train.datetime.min(),train.datetime.max(),len(train)); print('Teste',test.datetime.min(),test.datetime.max(),len(test))


In [ ]:
params={'n_estimators':[100,200,300,500], 'max_depth':[4,8,12,None], 'min_samples_leaf':[1,2,5,10]} # 3 hiperparâmetros
search=RandomizedSearchCV(RandomForestClassifier(class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1), params, n_iter=10, scoring='f1', cv=TimeSeriesSplit(5), random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
search.fit(X_train,y_train)
print('Melhores parâmetros:',search.best_params_,'CV F1:',search.best_score_)
results=pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
display(results[['params','mean_train_score','mean_test_score','std_test_score']])


In [ ]:
model=search.best_estimator_
def metrics(X,y,label):
    pred=model.predict(X); return {'conjunto':label,'accuracy':accuracy_score(y,pred),'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0)}
summary=pd.DataFrame([metrics(X_train,y_train,'treino'),metrics(X_test,y_test,'teste')]); display(summary)
pred=model.predict(X_test); print(classification_report(y_test,pred,zero_division=0))
sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cmap='Blues'); plt.title('Matriz de confusão — teste'); plt.xlabel('Previsto'); plt.ylabel('Real'); plt.show()


In [ ]:
importance=permutation_importance(model,X_test,y_test,n_repeats=10,random_state=RANDOM_STATE,scoring='f1')
pd.Series(importance.importances_mean,index=features).sort_values().plot.barh(title='Importância por permutação — teste'); plt.show()
print('Gap F1 treino-teste:',round(summary.loc[0,'f1']-summary.loc[1,'f1'],3))


## Conclusão a preencher após execução

Copie as métricas reais para `RELATORIO_ML.md`. Discuta falsos negativos (alerta perdido), falsos positivos (fadiga de alerta), desbalanceamento e gap treino–teste. Não afirme previsão clínica: o alvo é uma regra operacional experimental e alertas oficiais prevalecem.